In [ ]:
!pip install -qU langchain_community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
file_path="https://www.bu.edu/lernet/artemis/years/2011/slides/python.pdf"
loader=PyPDFLoader(file_path)
doc=loader.load()
print(doc[0].metadata)

/tmp/ipykernel_7673/873980537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


{'producer': 'Microsoft® Office PowerPoint® 2007', 'creator': 'Microsoft® Office PowerPoint® 2007', 'creationdate': 'D:20110801202339', 'title': 'Slide 1', 'author': 'Dyuchan', 'moddate': 'D:20110801202339', 'source': 'https://www.bu.edu/lernet/artemis/years/2011/slides/python.pdf', 'total_pages': 91, 'page': 0, 'page_label': '1'}


In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=200,
)
all_splits = text_splitter.split_documents(doc)
print(all_splits)
print(f"Paper split into {len(all_splits)} sub-documents.")
print(f"Metadata: {all_splits[0].metadata}")

[Document(metadata={'producer': 'Microsoft® Office PowerPoint® 2007', 'creator': 'Microsoft® Office PowerPoint® 2007', 'creationdate': 'D:20110801202339', 'title': 'Slide 1', 'author': 'Dyuchan', 'moddate': 'D:20110801202339', 'source': 'https://www.bu.edu/lernet/artemis/years/2011/slides/python.pdf', 'total_pages': 91, 'page': 1, 'page_label': '2'}, page_content='Languages\n• Solve problems using a computer, give the \ncomputer instructions.\n• Remember our diaper-changing exercise?'), Document(metadata={'producer': 'Microsoft® Office PowerPoint® 2007', 'creator': 'Microsoft® Office PowerPoint® 2007', 'creationdate': 'D:20110801202339', 'title': 'Slide 1', 'author': 'Dyuchan', 'moddate': 'D:20110801202339', 'source': 'https://www.bu.edu/lernet/artemis/years/2011/slides/python.pdf', 'total_pages': 91, 'page': 2, 'page_label': '3'}, page_content='Talk the talk\n• Speak its language\n– High-level: Python, C++, Java\n– Low-level: machine language, computers can only \nexecute these\n– Hig

In [ ]:
! pip install -qU langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
!pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
  collection_name="research_collection",
  embedding_function=embedding_model,
  persist_directory="./chroma_langchain_db"
)
document_ids = vector_store.add_documents(documents=all_splits)

sample = vector_store.get(limit=1, include=["embeddings", "documents"])
print(sample)
print(document_ids[:3])

{'ids': ['9dbcbaba-62a2-4220-a706-53ef5415ce11'], 'embeddings': array([[ 2.73822453e-02,  5.50563112e-02, -5.09197004e-02,
         8.41385126e-03, -3.92316431e-02,  2.37492081e-02,
         3.37015651e-02, -2.40944442e-03,  6.20694991e-05,
         4.33744956e-03,  5.94641976e-02, -4.37449142e-02,
         2.33159829e-02,  5.39281545e-03,  3.91083173e-02,
        -7.27726817e-02,  9.15521011e-03, -3.21061015e-02,
        -3.74768674e-02,  5.94387297e-03, -2.32886765e-02,
         1.81336291e-02, -5.46427704e-02,  1.96255241e-02,
         2.02846006e-02, -4.74788658e-02, -1.98662058e-02,
        -4.71781977e-02, -8.63513630e-03,  1.52068762e-02,
         3.28676738e-02,  2.22789198e-02,  8.58067162e-03,
        -2.70984434e-02,  1.47622166e-06, -2.88506784e-02,
         1.55771961e-02,  2.48373370e-03, -4.46791090e-02,
        -1.25906337e-02,  3.25701456e-03,  1.48731517e-02,
         6.75011333e-03,  4.32041194e-03, -4.44545038e-02,
         6.57975581e-03,  6.52551204e-02,  8.721762

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
def retrieve_context(query: str, k: int = 2):
  retrieved_docs = vector_store.similarity_search(query, k=k)

  docs_content = ""
  for doc in retrieved_docs:
    docs_content += f"Source: {doc.metadata}\n"
    docs_content += f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get('Gemini_api_key')
model = init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key=api_key,
)

def docu_chat(user_query):
  context, source_docs = retrieve_context(user_query, k=2)
  system_message = f"""You are a helpful chatbot.
                     Use only the following pieces of context to answer the
                     question. Don't makeup any new information: {context} """

  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_query}
  ]
  response = model.invoke(messages)
  return {
  "answer": response.content,
  "source_documents": source_docs,
  "context_used": context
}
result = docu_chat( "Explain what is the use of decoders in transformers?")
print(result)
print(result["answer"])

{'answer': 'I apologize, but the provided context does not contain information about the use of decoders in transformers. The context only discusses accumulator variables and their purposes (counting, keeping score, and debugging).', 'source_documents': [Document(id='210f63b9-7624-4c52-abc4-11dc7f9b32e4', metadata={'creator': 'Microsoft® Office PowerPoint® 2007', 'creationdate': 'D:20110801202339', 'author': 'Dyuchan', 'page_label': '50', 'moddate': 'D:20110801202339', 'page': 49, 'title': 'Slide 1', 'total_pages': 91, 'producer': 'Microsoft® Office PowerPoint® 2007', 'source': 'https://www.bu.edu/lernet/artemis/years/2011/slides/python.pdf'}, page_content='Accumulator Variables\nWhat is the point of accumulator variables?\n-Counting.\n-keeping score (affects program does).\n-debugging.'), Document(id='a3c0bbf2-0da7-467d-aba3-50952df89a4d', metadata={'author': 'Dyuchan', 'moddate': 'D:20110801202339', 'page_label': '50', 'title': 'Slide 1', 'page': 49, 'creator': 'Microsoft® Office Pow